<a href="https://colab.research.google.com/github/ciertou/handsonllms/blob/main/tensor_logic_org.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [2]:
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_size, max_len):
        super(TransformerEmbedding, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_len, embed_size)

    def forward(self, X):
        # X is of shape (batch_size, sequence_length)
        batch_size, seq_len = X.size()

        token_embeddings = self.token_embedding(X)  # Shape: (batch_size, seq_len, embed_size)
        position_ids = torch.arange(seq_len, device=X.device).unsqueeze(0).repeat(batch_size, 1)
        position_embeddings = self.position_embedding(position_ids)  # Shape: (batch_size, seq_len, embed_size)

        return token_embeddings + position_embeddings  # Residual stream


In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_size, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_size = embed_size
        self.d_k = embed_size // num_heads  # dimension of each attention head

        self.query_linear = nn.Linear(embed_size, embed_size)
        self.key_linear = nn.Linear(embed_size, embed_size)
        self.value_linear = nn.Linear(embed_size, embed_size)
        self.out_linear = nn.Linear(embed_size, embed_size)

    def forward(self, x):
        batch_size, seq_len, embed_size = x.size()

        # Linear projections
        queries = self.query_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k)
        keys = self.key_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k)
        values = self.value_linear(x).view(batch_size, seq_len, self.num_heads, self.d_k)

        # Transpose to get the dimensions (batch_size, num_heads, seq_len, d_k)
        queries = queries.transpose(1, 2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        # Scaled dot-product attention
        attn_weights = torch.matmul(queries, keys.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(attn_weights, dim=-1)

        # Weighted sum of values
        attn_output = torch.matmul(attn_weights, values)

        # Concatenate the heads and put through output linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, embed_size)
        output = self.out_linear(attn_output)

        return output


In [4]:
class FeedForward(nn.Module):
    def __init__(self, embed_size, hidden_size):
        super(FeedForward, self).__init__()
        self.fc1 = nn.Linear(embed_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, embed_size)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


In [5]:
class TransformerLayer(nn.Module):
    def __init__(self, embed_size, num_heads, hidden_size):
        super(TransformerLayer, self).__init__()
        self.attn = MultiHeadAttention(embed_size, num_heads)
        self.ffn = FeedForward(embed_size, hidden_size)
        self.layer_norm1 = nn.LayerNorm(embed_size)
        self.layer_norm2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        # Attention with residual connection
        attn_output = self.attn(x)
        x = self.layer_norm1(x + attn_output)

        # Feedforward network with residual connection
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)

        return x


In [6]:
class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_layers, hidden_size, max_len):
        super(TransformerModel, self).__init__()
        self.embedding = TransformerEmbedding(vocab_size, embed_size, max_len)
        self.transformer_layers = nn.ModuleList([TransformerLayer(embed_size, num_heads, hidden_size) for _ in range(num_layers)])
        self.fc_out = nn.Linear(embed_size, vocab_size)  # Output layer (for generating words)

    def forward(self, X):
        x = self.embedding(X)

        # Pass through all transformer layers
        for layer in self.transformer_layers:
            x = layer(x)

        # Final output
        output = self.fc_out(x)

        # Apply softmax to get probability distribution for each token
        return F.softmax(output, dim=-1)


In [7]:
# Hyperparameters
vocab_size = 10000  # Example vocabulary size
embed_size = 512    # Embedding size
num_heads = 8       # Number of attention heads
num_layers = 6      # Number of transformer layers
hidden_size = 2048  # Feedforward hidden size
max_len = 100       # Maximum sequence length
batch_size = 32     # Batch size
seq_len = 50        # Sequence length

# Create random input (batch_size, seq_len)
X = torch.randint(0, vocab_size, (batch_size, seq_len))

# Instantiate model
model = TransformerModel(vocab_size, embed_size, num_heads, num_layers, hidden_size, max_len)

# Forward pass
output = model(X)
print(output.shape)  # Expected shape: (batch_size, seq_len, vocab_size)


torch.Size([32, 50, 10000])
